# DGCNN Autoencoder + ReAct Agentic AI -- Maize Plant Analysis

**Pipeline**
1. DGCNN Autoencoder -- best reconstruction model (point-cloud native, 4-block multi-scale)
2. Yield-prediction MLP trained on latent codes -- exposed as an agent tool
3. Latent interpolation tool -- morph between two plants in latent space
4. Memory module -- the agent remembers past queries and results across turns
5. Full ReAct Agent -- Thought / Action / Observation loop with all tools wired together


In [ ]:
!pip install open3d plotly huggingface_hub numpy scipy wandb -q
!pip install torch scikit-learn -q

import open3d as o3d
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import wandb, os, math, time, zipfile, json, re
from collections import deque
from sklearn.decomposition import PCA
from huggingface_hub import hf_hub_download
from torch.utils.data import Dataset, DataLoader, Subset, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.5 MB/s eta 0:00:00
Using device: cuda


In [ ]:
zip_path = hf_hub_download(
    repo_id='BGLab/AgriField3D',
    filename='datasets/FielGrwon_ZeaMays_RawPCD_10k.zip',
    repo_type='dataset', local_dir='./data'
)
extract_dir = './data/RawPCD_10k'
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)
ply_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(extract_dir)
    for f in files if f.endswith('.ply')
])
print(f'Total plants found: {len(ply_files)}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


datasets/FielGrwon_ZeaMays_RawPCD_10k.zi(…):   0%|          | 0.00/218M [00:00<?, ?B/s]

Total plants found: 1045


In [ ]:
NUM_POINTS = 1024
TRAIN_SIZE = 500

def preprocess(ply_path, num_points=NUM_POINTS):
    pcd = o3d.io.read_point_cloud(ply_path)
    pts = np.asarray(pcd.points, dtype=np.float32)
    if pts.shape[0] == 0: return None
    N   = pts.shape[0]
    idx = (np.random.choice(N, num_points, replace=False)
           if N >= num_points else np.random.choice(N, num_points, replace=True))
    pts  = pts[idx]
    pts -= pts.mean(axis=0)
    pts /= (np.linalg.norm(pts, axis=1).max() + 1e-8)
    return pts

class MaizeDataset(Dataset):
    def __init__(self, ply_files):
        self.samples, self.paths = [], []
        print(f'Loading {len(ply_files)} files ...')
        for i, path in enumerate(ply_files):
            try:
                pts = preprocess(path)
                if pts is not None:
                    self.samples.append(torch.tensor(pts))
                    self.paths.append(path)
            except Exception as e:
                print(f'  [skip] {os.path.basename(path)}: {e}')
            if (i+1) % 20 == 0:
                print(f'  {i+1}/{len(ply_files)} -- {len(self.samples)} valid')
        print(f'Dataset ready: {len(self.samples)} samples')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx], idx

full_dataset  = MaizeDataset(ply_files)
total         = len(full_dataset)
all_indices   = np.random.permutation(total)
train_indices = all_indices[:TRAIN_SIZE].tolist()
test_indices  = all_indices[TRAIN_SIZE:].tolist()
train_dataset = Subset(full_dataset, train_indices)
test_dataset  = Subset(full_dataset, test_indices)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True,
                           drop_last=True, num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=8, shuffle=False,
                           drop_last=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_dataset)}  |  Test: {len(test_dataset)}')


Loading 1045 files ...
  20/1045 -- 20 valid
  40/1045 -- 40 valid
  60/1045 -- 60 valid
  80/1045 -- 80 valid
  100/1045 -- 100 valid
  120/1045 -- 120 valid
  140/1045 -- 140 valid
  160/1045 -- 160 valid
  180/1045 -- 180 valid
  200/1045 -- 200 valid
  220/1045 -- 220 valid
  240/1045 -- 240 valid
  260/1045 -- 260 valid
  280/1045 -- 280 valid
  300/1045 -- 300 valid
  320/1045 -- 320 valid
  340/1045 -- 340 valid
  360/1045 -- 360 valid
  380/1045 -- 380 valid
  400/1045 -- 400 valid
  420/1045 -- 420 valid
  440/1045 -- 440 valid
  460/1045 -- 460 valid
  480/1045 -- 480 valid
  500/1045 -- 500 valid
  520/1045 -- 520 valid
  540/1045 -- 540 valid
  560/1045 -- 560 valid
  580/1045 -- 580 valid
  600/1045 -- 600 valid
  620/1045 -- 620 valid
  640/1045 -- 640 valid
  660/1045 -- 660 valid
  680/1045 -- 680 valid
  700/1045 -- 700 valid
  720/1045 -- 720 valid
  740/1045 -- 740 valid
  760/1045 -- 760 valid
  780/1045 -- 780 valid
  800/1045 -- 800 valid
  820/1045 -- 820 valid
 

In [ ]:
# DGCNN Autoencoder -- chosen over Voxel AE because:
#  * Works directly on point clouds (no voxelisation quantisation loss)
#  * 4-block multi-scale EdgeConv encoder captures fine plant geometry
#  * Decoder without BatchNorm preserves per-plant shape diversity
#  * Chamfer Distance loss directly penalises geometric error
#  * Lower memory footprint at same latent dim

def knn(x, k):
    inner             = -2 * torch.matmul(x.transpose(2, 1), x)
    xx                = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    return pairwise_distance.topk(k=k, dim=-1)[1]

def get_graph_feature(x, k=16):
    B, C, N  = x.size()
    idx      = knn(x, k)
    idx_base = torch.arange(0, B, device=x.device).view(-1, 1, 1) * N
    idx      = (idx + idx_base).view(-1)
    x_t      = x.transpose(2, 1).contiguous()
    feature  = x_t.view(B*N, -1)[idx, :].view(B, N, k, C)
    x_t      = x_t.view(B, N, 1, C).repeat(1, 1, k, 1)
    return torch.cat((feature - x_t, x_t), dim=3).permute(0, 3, 1, 2)

class DGCNN_Encoder(nn.Module):
    def __init__(self, k=16, latent_dim=256):
        super().__init__()
        self.k = k
        self.conv1 = nn.Sequential(nn.Conv2d(6,   64,  1, bias=False), nn.BatchNorm2d(64),  nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(128, 64,  1, bias=False), nn.BatchNorm2d(64),  nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(128, 128, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(256, 256, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2))
        self.fc    = nn.Sequential(
            nn.Linear(512, 512), nn.BatchNorm1d(512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, latent_dim)
        )
    def forward(self, x):
        B = x.size(0); k = self.k
        x0 = self.conv1(get_graph_feature(x,  k)).max(-1)[0]
        x1 = self.conv2(get_graph_feature(x0, k)).max(-1)[0]
        x2 = self.conv3(get_graph_feature(x1, k)).max(-1)[0]
        x3 = self.conv4(get_graph_feature(x2, k)).max(-1)[0]
        xg = F.adaptive_max_pool1d(torch.cat((x0,x1,x2,x3), dim=1), 1).view(B, -1)
        return self.fc(xg)

class DGCNN_Decoder(nn.Module):
    # No BatchNorm -- preserves per-plant shape diversity
    def __init__(self, latent_dim=256, num_points=NUM_POINTS):
        super().__init__()
        self.num_points = num_points
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.ReLU(),
            nn.Linear(512, 1024),       nn.ReLU(),
            nn.Linear(1024, num_points * 3)
        )
    def forward(self, z): return self.net(z).view(-1, self.num_points, 3)

class DGCNN_Autoencoder(nn.Module):
    def __init__(self, k=16, latent_dim=256, num_points=NUM_POINTS):
        super().__init__()
        self.encoder = DGCNN_Encoder(k=k, latent_dim=latent_dim)
        self.decoder = DGCNN_Decoder(latent_dim=latent_dim, num_points=num_points)
    def forward(self, x):
        z = self.encoder(x.transpose(2, 1))
        return self.decoder(z), z
    def encode(self, x): return self.encoder(x.transpose(2, 1))
    def decode(self, z): return self.decoder(z)

def chamfer_distance(pred, target):
    d1 = (pred.unsqueeze(2) - target.unsqueeze(1)).pow(2).sum(-1)
    d2 = (target.unsqueeze(2) - pred.unsqueeze(1)).pow(2).sum(-1)
    return d1.min(2)[0].mean() + d2.min(2)[0].mean()

print('DGCNN model defined.')
print(f'Params: {sum(p.numel() for p in DGCNN_Autoencoder().parameters()):,}')


DGCNN model defined.
Params: 4,292,224


In [ ]:
CKPT_DIR = './checkpoints_dgcnn'
os.makedirs(CKPT_DIR, exist_ok=True)

def save_checkpoint(model, optimizer, epoch, loss, tag='latest'):
    path = os.path.join(CKPT_DIR, f'dgcnn_{tag}.pt')
    torch.save({'epoch':epoch,'model_state':model.state_dict(),
                'optim_state':optimizer.state_dict(),'loss':loss}, path)
    return path

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    if optimizer and 'optim_state' in ckpt:
        optimizer.load_state_dict(ckpt['optim_state'])
    print(f'  Loaded -- epoch {ckpt["epoch"]} | loss {ckpt["loss"]:.6f}')
    return ckpt


In [ ]:
CONFIG = dict(
    epochs=500, batch_size=8, lr=1e-4, k=16,
    latent_dim=256, num_points=NUM_POINTS,
    train_size=TRAIN_SIZE, checkpoint_every=50,
)

wandb.init(project='maize-dgcnn-agent', config=CONFIG, name='dgcnn-ae-500ep')

ae_model = DGCNN_Autoencoder(
    k=CONFIG['k'], latent_dim=CONFIG['latent_dim'], num_points=CONFIG['num_points']
).to(device)

optimizer = torch.optim.Adam(ae_model.parameters(), lr=CONFIG['lr'])
wandb.watch(ae_model, log='all', log_freq=20)

best_loss, train_losses = float('inf'), []
t0 = time.time()

for epoch in range(1, CONFIG['epochs'] + 1):
    ae_model.train()
    epoch_loss = 0.0
    for batch, _ in train_loader:
        batch = batch.to(device, non_blocking=True)
        optimizer.zero_grad()
        recon, _ = ae_model(batch)
        loss = chamfer_distance(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)
    log_loss = math.log10(epoch_loss + 1e-10)
    wandb.log({'epoch':epoch,'train_loss':epoch_loss,
               'train_loss_log':log_loss,'lr':CONFIG['lr'],
               'elapsed_min':(time.time()-t0)/60})

    if epoch % 50 == 0:
        ae_model.eval()
        sample = next(iter(train_loader))[0][:1].to(device)
        with torch.no_grad():
            s_recon, _ = ae_model(sample)
        o, r = sample[0].cpu().numpy(), s_recon[0].cpu().numpy()
        wandb.log({
            f'train_original_ep{epoch}': wandb.Object3D(np.hstack([o, np.tile([0,120,255],(len(o),1))])),
            f'train_recon_ep{epoch}':    wandb.Object3D(np.hstack([r, np.tile([255,80,0],(len(r),1))])),
        })
        ae_model.train()

    if epoch % CONFIG['checkpoint_every'] == 0:
        path = save_checkpoint(ae_model, optimizer, epoch, epoch_loss, 'latest')
        wandb.save(path)
        print(f'Epoch {epoch:4d} | Loss {epoch_loss:.6f} | {(time.time()-t0)/60:.1f} min')

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        save_checkpoint(ae_model, optimizer, epoch, epoch_loss, 'best')

print(f'Training done. Best loss: {best_loss:.6f}')

# Dual loss curve
epochs_x = list(range(1, len(train_losses)+1))
fig_dual = make_subplots(rows=1, cols=2, subplot_titles=('Loss -- Linear','Loss -- Log'))
for col, use_log in [(1,False),(2,True)]:
    fig_dual.add_trace(go.Scatter(x=epochs_x, y=train_losses, mode='lines',
        line=dict(color='#636EFA' if col==1 else '#FF7F0E', width=2),
        showlegend=(col==1), name='Train Loss'), row=1, col=col)
    if use_log: fig_dual.update_yaxes(type='log', title_text='CD (log)', row=1, col=col)
    else:       fig_dual.update_yaxes(title_text='CD', row=1, col=col)
    fig_dual.update_xaxes(title_text='Epoch', row=1, col=col)
fig_dual.update_layout(title='DGCNN Training Loss', template='plotly_dark', height=420)
fig_dual.show()
wandb.log({'loss_curve': wandb.Plotly(fig_dual)})
wandb.finish()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ghosalsohom2003 (ghosalsohom2003-own-use) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch   50 | Loss 0.005561 | 5.0 min
Epoch  100 | Loss 0.003793 | 9.9 min
Epoch  150 | Loss 0.002655 | 14.8 min
Epoch  200 | Loss 0.001997 | 19.7 min
Epoch  250 | Loss 0.001576 | 24.6 min
Epoch  300 | Loss 0.001238 | 29.5 min
Epoch  350 | Loss 0.001035 | 34.4 min
Epoch  400 | Loss 0.000907 | 39.2 min
Epoch  450 | Loss 0.000805 | 44.1 min
Epoch  500 | Loss 0.000711 | 49.0 min
Training done. Best loss: 0.000700


elapsed_min,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇██
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▆▅▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss_log,█▆▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁
elapsed_min,48.98319
epoch,500
lr,0.0001
train_loss,0.00071
train_loss_log,-3.14843


In [ ]:
# Load best checkpoint
ae_model = DGCNN_Autoencoder(
    k=CONFIG['k'], latent_dim=CONFIG['latent_dim'], num_points=CONFIG['num_points']
).to(device)
load_checkpoint(os.path.join(CKPT_DIR, 'dgcnn_best.pt'), ae_model)
ae_model.eval()

all_latents, all_indices_order = [], []
with torch.no_grad():
    for batch, idxs in DataLoader(full_dataset, batch_size=16):
        z = ae_model.encode(batch.to(device))
        all_latents.append(z.cpu().numpy())
        all_indices_order.extend(idxs.tolist())

all_latents = np.concatenate(all_latents, axis=0)   # [N, 256]
print(f'Extracted latents: {all_latents.shape}')

# Proxy yield labels derived from latent geometry
# (real dataset lacks yield annotations; PCA-1 correlates with plant size/volume)
pca_1d = PCA(n_components=1).fit_transform(all_latents).squeeze()
yield_labels = (
    2.5
    + 1.5 * (pca_1d - pca_1d.min()) / (pca_1d.max() - pca_1d.min())
    + 0.3 * np.random.randn(len(all_latents))
).astype(np.float32)
print(f'Yield range: {yield_labels.min():.2f} -- {yield_labels.max():.2f} t/ha')
print(f'Yield mean:  {yield_labels.mean():.2f} t/ha')


  Loaded -- epoch 497 | loss 0.000700
Extracted latents: (1045, 256)
Yield range: 1.99 -- 4.58 t/ha
Yield mean:  3.32 t/ha


In [ ]:
class YieldMLP(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),         nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, z): return self.net(z).squeeze(-1)

# Build train/val split
Z_tensor = torch.tensor(all_latents, dtype=torch.float32)
Y_tensor = torch.tensor(yield_labels, dtype=torch.float32)
perm = torch.randperm(len(Z_tensor))
n_tr = int(0.8 * len(Z_tensor))
Z_tr, Y_tr = Z_tensor[perm[:n_tr]], Y_tensor[perm[:n_tr]]
Z_va, Y_va = Z_tensor[perm[n_tr:]], Y_tensor[perm[n_tr:]]

yield_loader_tr = DataLoader(TensorDataset(Z_tr, Y_tr), batch_size=32, shuffle=True)
yield_loader_va = DataLoader(TensorDataset(Z_va, Y_va), batch_size=32)

yield_mlp  = YieldMLP(latent_dim=CONFIG['latent_dim']).to(device)
yield_opt  = torch.optim.Adam(yield_mlp.parameters(), lr=1e-3, weight_decay=1e-4)
yield_sch  = torch.optim.lr_scheduler.CosineAnnealingLR(yield_opt, T_max=200, eta_min=1e-5)
loss_fn    = nn.MSELoss()

CKPT_YIELD = './checkpoints_yield'
os.makedirs(CKPT_YIELD, exist_ok=True)
best_val_rmse = float('inf')

print('Training yield MLP (200 epochs) ...')
for ep in range(1, 201):
    yield_mlp.train()
    for zb, yb in yield_loader_tr:
        zb, yb = zb.to(device), yb.to(device)
        yield_opt.zero_grad()
        loss_fn(yield_mlp(zb), yb).backward()
        yield_opt.step()
    yield_sch.step()
    if ep % 20 == 0:
        yield_mlp.eval()
        preds, trues = [], []
        with torch.no_grad():
            for zb, yb in yield_loader_va:
                preds.append(yield_mlp(zb.to(device)).cpu())
                trues.append(yb)
        preds, trues = torch.cat(preds).numpy(), torch.cat(trues).numpy()
        rmse = np.sqrt(((preds-trues)**2).mean())
        mae  = np.abs(preds-trues).mean()
        print(f'  Epoch {ep:3d} | Val RMSE {rmse:.4f} | Val MAE {mae:.4f} t/ha')
        if rmse < best_val_rmse:
            best_val_rmse = rmse
            torch.save(yield_mlp.state_dict(), os.path.join(CKPT_YIELD, 'yield_mlp_best.pt'))

yield_mlp.load_state_dict(torch.load(os.path.join(CKPT_YIELD,'yield_mlp_best.pt'), map_location=device))
yield_mlp.eval()
print(f'Best val RMSE: {best_val_rmse:.4f} t/ha')


Training yield MLP (200 epochs) ...
  Epoch  20 | Val RMSE 0.5609 | Val MAE 0.4772 t/ha
  Epoch  40 | Val RMSE 0.5487 | Val MAE 0.4574 t/ha
  Epoch  60 | Val RMSE 0.5285 | Val MAE 0.4337 t/ha
  Epoch  80 | Val RMSE 0.5135 | Val MAE 0.4290 t/ha
  Epoch 100 | Val RMSE 0.4500 | Val MAE 0.3724 t/ha
  Epoch 120 | Val RMSE 0.4743 | Val MAE 0.3892 t/ha
  Epoch 140 | Val RMSE 0.4623 | Val MAE 0.3797 t/ha
  Epoch 160 | Val RMSE 0.4603 | Val MAE 0.3766 t/ha
  Epoch 180 | Val RMSE 0.4611 | Val MAE 0.3813 t/ha
  Epoch 200 | Val RMSE 0.4478 | Val MAE 0.3674 t/ha
Best val RMSE: 0.4478 t/ha


In [ ]:
# Pre-build latent index for O(1) lookup
LATENT_INDEX = {idx: all_latents[i] for i, idx in enumerate(all_indices_order)}

# ── Tool 1: encode_plant ──────────────────────────────────────────────
def tool_encode_plant(plant_idx: int) -> dict:
    '''Encode a plant (by dataset index) to its 256-d latent vector.'''
    if plant_idx not in LATENT_INDEX:
        return {'error': f'plant_idx {plant_idx} not found'}
    z_np = LATENT_INDEX[plant_idx]
    pts, _ = full_dataset[plant_idx]
    pts_t  = pts.unsqueeze(0).to(device)
    with torch.no_grad():
        z_t   = ae_model.encode(pts_t)
        recon = ae_model.decode(z_t)
        cd    = chamfer_distance(recon, pts_t).item()
    return {'plant_idx': plant_idx, 'latent': z_np.tolist(), 'chamfer_cd': round(cd, 6)}

# ── Tool 2: decode_latent ─────────────────────────────────────────────
def tool_decode_latent(latent: list) -> dict:
    '''Decode a latent vector into a 3D point cloud.'''
    z = torch.tensor(latent, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        pts = ae_model.decode(z)[0].cpu().numpy()
    return {'point_cloud': pts.tolist(), 'num_points': len(pts),
            'bbox_min': pts.min(axis=0).tolist(), 'bbox_max': pts.max(axis=0).tolist()}

# ── Tool 3: predict_yield (with MC-Dropout uncertainty) ───────────────
def tool_predict_yield(latent: list) -> dict:
    '''Predict maize yield (t/ha) + 95% CI via Monte Carlo dropout.'''
    z = torch.tensor(latent, dtype=torch.float32).unsqueeze(0).to(device)
    yield_mlp.train()   # enable dropout for MC sampling
    preds = []
    with torch.no_grad():
        for _ in range(20):
            preds.append(yield_mlp(z).item())
    yield_mlp.eval()
    mu, std = float(np.mean(preds)), float(np.std(preds))
    return {
        'yield_t_ha':          round(mu, 3),
        'uncertainty_std':     round(std, 4),
        'confidence_interval': [round(mu - 2*std, 3), round(mu + 2*std, 3)],
        'interpretation':      ('High yield' if mu > 3.5 else
                                'Medium yield' if mu > 2.5 else 'Low yield'),
    }

# ── Tool 4: interpolate_latents ───────────────────────────────────────
def tool_interpolate_latents(plant_idx_a: int, plant_idx_b: int, steps: int = 5) -> dict:
    '''Linearly interpolate in latent space -- virtual morphing / breeding.
    Returns step-by-step yields and geometry metrics along the trajectory.'''
    if plant_idx_a not in LATENT_INDEX or plant_idx_b not in LATENT_INDEX:
        return {'error': 'One or both plant indices not found'}
    z_a = torch.tensor(LATENT_INDEX[plant_idx_a], dtype=torch.float32).to(device)
    z_b = torch.tensor(LATENT_INDEX[plant_idx_b], dtype=torch.float32).to(device)
    results = []
    for i, alpha in enumerate(np.linspace(0, 1, steps)):
        z_i = (1 - alpha) * z_a + alpha * z_b
        with torch.no_grad():
            pts = ae_model.decode(z_i.unsqueeze(0))[0].cpu().numpy()
        yr = tool_predict_yield(z_i.cpu().tolist())
        results.append({'step': i, 'alpha': round(float(alpha), 3),
                        'yield_t_ha': yr['yield_t_ha'],
                        'uncertainty': yr['uncertainty_std'],
                        'bbox_height': round(float(pts[:,2].max()-pts[:,2].min()), 4),
                        'latent_norm': round(float(z_i.norm().item()), 4)})
    return {'plant_a': plant_idx_a, 'plant_b': plant_idx_b, 'steps': steps,
            'trajectory': results,
            'yield_range': [round(min(r['yield_t_ha'] for r in results), 3),
                            round(max(r['yield_t_ha'] for r in results), 3)]}

# ── Tool 5: compare_plants ────────────────────────────────────────────
def tool_compare_plants(plant_idx_a: int, plant_idx_b: int) -> dict:
    '''Compare two plants: Chamfer distance, cosine similarity, yield diff.'''
    if plant_idx_a not in LATENT_INDEX or plant_idx_b not in LATENT_INDEX:
        return {'error': 'One or both plant indices not found'}
    z_a = torch.tensor(LATENT_INDEX[plant_idx_a], dtype=torch.float32).to(device)
    z_b = torch.tensor(LATENT_INDEX[plant_idx_b], dtype=torch.float32).to(device)
    cos_sim = F.cosine_similarity(z_a.unsqueeze(0), z_b.unsqueeze(0)).item()
    with torch.no_grad():
        pc_a = ae_model.decode(z_a.unsqueeze(0))
        pc_b = ae_model.decode(z_b.unsqueeze(0))
        cd   = chamfer_distance(pc_a, pc_b).item()
    y_a = tool_predict_yield(z_a.cpu().tolist())['yield_t_ha']
    y_b = tool_predict_yield(z_b.cpu().tolist())['yield_t_ha']
    return {'plant_a': plant_idx_a, 'plant_b': plant_idx_b,
            'chamfer_distance': round(cd, 6),
            'cosine_similarity': round(cos_sim, 4),
            'yield_a_t_ha': y_a, 'yield_b_t_ha': y_b,
            'yield_diff_t_ha': round(y_b - y_a, 3),
            'morphology_similar': cos_sim > 0.85}

# ── Tool 6: latent_statistics ─────────────────────────────────────────
def tool_latent_statistics(latent: list) -> dict:
    '''Analyse a latent vector: norm, activations, percentile rank in dataset.'''
    z         = np.array(latent)
    all_norms = np.linalg.norm(all_latents, axis=1)
    my_norm   = float(np.linalg.norm(z))
    pct       = float(np.mean(all_norms < my_norm) * 100)
    mean_z    = all_latents.mean(axis=0)
    cos_to_mean = float(np.dot(z, mean_z) /
                        (np.linalg.norm(z)*np.linalg.norm(mean_z) + 1e-8))
    return {'latent_norm': round(my_norm, 4), 'norm_percentile': round(pct, 1),
            'mean_activation': round(float(z.mean()), 4),
            'std_activation':  round(float(z.std()),  4),
            'max_activation':  round(float(z.max()),  4),
            'min_activation':  round(float(z.min()),  4),
            'cosine_to_mean_plant': round(cos_to_mean, 4),
            'interpretation': ('Unusually large plant' if pct > 85 else
                               'Unusually small plant' if pct < 15 else
                               'Typical plant morphology')}

TOOLS = {
    'encode_plant':        tool_encode_plant,
    'decode_latent':       tool_decode_latent,
    'predict_yield':       tool_predict_yield,
    'interpolate_latents': tool_interpolate_latents,
    'compare_plants':      tool_compare_plants,
    'latent_statistics':   tool_latent_statistics,
}
print('All 6 agent tools registered.')


All 6 agent tools registered.


In [ ]:
class AgentMemory:
    '''
    Episodic memory for the ReAct agent.
    Stores full history (capped at max_history turns),
    condensed facts, and a tool-call log.
    Supports keyword retrieval over facts.
    '''
    def __init__(self, max_history=100):
        self.max_history = max_history
        self.history:  deque = deque(maxlen=max_history)
        self.facts:    list  = []
        self.tool_log: list  = []

    def add_turn(self, role, content):
        self.history.append({'role':role,'content':content,'time':time.strftime('%H:%M:%S')})

    def add_fact(self, fact):
        self.facts.append({'fact':fact,'time':time.strftime('%H:%M:%S')})

    def log_tool(self, tool_name, args, result_summary):
        self.tool_log.append({'tool':tool_name,'args':args,
                              'summary':result_summary,'time':time.strftime('%H:%M:%S')})

    def get_recent_history(self, n=6):  return list(self.history)[-n:]

    def search_facts(self, keyword):
        kw = keyword.lower()
        return [f for f in self.facts if kw in f['fact'].lower()]

    def get_tool_log(self, tool_name=None):
        return self.tool_log if tool_name is None else \
               [t for t in self.tool_log if t['tool'] == tool_name]

    def format_context(self, n_recent=4):
        lines = ['=== AGENT MEMORY ===']
        if self.facts:
            lines.append('Key findings:')
            for f in self.facts[-5:]: lines.append(f'  [{f["time"]}] {f["fact"]}')
        if self.tool_log:
            lines.append('Recent tool calls:')
            for t in self.tool_log[-3:]:
                lines.append(f'  [{t["time"]}] {t["tool"]}({t["args"]}) -> {t["summary"][:80]}')
        lines.append('=== END MEMORY ===')
        return '\n'.join(lines)

    def clear(self):
        self.history.clear(); self.facts.clear(); self.tool_log.clear()

    def __repr__(self):
        return (f'AgentMemory(turns={len(self.history)},'
                f'facts={len(self.facts)},tool_calls={len(self.tool_log)})')

agent_memory = AgentMemory(max_history=100)
print('Memory module ready:', agent_memory)


Memory module ready: AgentMemory(turns=0,facts=0,tool_calls=0)


In [ ]:
def parse_action(text):
    m = re.search(r'Action:\s*(\w+)\((.*?)\)', text, re.DOTALL)
    if not m: return None, None
    tool_name, args_str = m.group(1).strip(), m.group(2).strip()
    kwargs = {}
    if args_str:
        for part in re.split(r',\s*(?![^\[]*\])', args_str):
            part = part.strip()
            if '=' in part:
                k, v = part.split('=', 1)
                try:    kwargs[k.strip()] = json.loads(v.strip())
                except: kwargs[k.strip()] = v.strip().strip('"').strip("'")
    return tool_name, kwargs


def react_step(thought, memory):
    memory.add_turn('thought', thought)
    tool_name, kwargs = parse_action(thought)
    if tool_name is None or tool_name not in TOOLS:
        obs = {'error': f'No valid Action found'}
        memory.add_turn('observation', str(obs))
        return None, obs
    print(f'  >> Calling: {tool_name}({list(kwargs.keys())})')
    try:
        result = TOOLS[tool_name](**kwargs)
    except Exception as e:
        result = {'error': str(e)}
    summary = str(result)[:120] + ('...' if len(str(result))>120 else '')
    memory.log_tool(tool_name, kwargs, summary)
    memory.add_turn('observation', json.dumps(result)[:300])
    return tool_name, result


def react_agent(query, memory, max_steps=8, verbose=True):
    '''
    ReAct loop: Thought -> Action -> Observation, with memory.
    The planner is rule-based (no external LLM API needed).
    Covers: yield prediction, comparison, interpolation, latent analysis.
    '''
    if verbose:
        print('='*60)
        print(f'QUERY: {query}')
        print('='*60)

    memory.add_turn('user', query)
    nums    = [int(x) for x in re.findall(r'\b(\d+)\b', query)]
    q_lower = query.lower()
    observations, prev_latent = {}, None

    # Build execution plan from query intent
    if any(w in q_lower for w in ['interpolat','morph','between','breed','transition']) and len(nums)>=2:
        a, b  = nums[0], nums[1]
        steps = nums[2] if len(nums)>2 else 5
        plan  = [
            f'Encode plant {a}.\nAction: encode_plant(plant_idx={a})',
            f'Encode plant {b}.\nAction: encode_plant(plant_idx={b})',
            f'Interpolate between {a} and {b}.\nAction: interpolate_latents(plant_idx_a={a}, plant_idx_b={b}, steps={steps})',
        ]
    elif any(w in q_lower for w in ['compar','differ','versus','vs']) and len(nums)>=2:
        a, b = nums[0], nums[1]
        plan = [
            f'Compare plants {a} and {b}.\nAction: compare_plants(plant_idx_a={a}, plant_idx_b={b})',
            f'Get latent stats for plant {a}.\nAction: encode_plant(plant_idx={a})',
        ]
    elif any(w in q_lower for w in ['yield','predict','forecast','produc']) and len(nums)>=1:
        a    = nums[0]
        plan = [
            f'Encode plant {a}.\nAction: encode_plant(plant_idx={a})',
            f'Predict yield.\nAction: predict_yield(latent=PREV_LATENT)',
            f'Latent statistics.\nAction: latent_statistics(latent=PREV_LATENT)',
        ]
    elif any(w in q_lower for w in ['latent','statistic','analys','encod']) and len(nums)>=1:
        a    = nums[0]
        plan = [
            f'Encode plant {a}.\nAction: encode_plant(plant_idx={a})',
            f'Analyse latent vector.\nAction: latent_statistics(latent=PREV_LATENT)',
        ]
    elif len(nums)>=1:
        a    = nums[0]
        plan = [
            f'Encode plant {a}.\nAction: encode_plant(plant_idx={a})',
            f'Predict yield.\nAction: predict_yield(latent=PREV_LATENT)',
            f'Latent statistics.\nAction: latent_statistics(latent=PREV_LATENT)',
        ]
    else:
        plan = ['Please provide a plant index.']

    # Execute plan
    for step_i, step_thought in enumerate(plan[:max_steps], 1):
        if 'PREV_LATENT' in step_thought and prev_latent is not None:
            step_thought = step_thought.replace('PREV_LATENT', json.dumps(prev_latent))
        if verbose:
            print(f'\n--- Step {step_i} ---')
            print(f'Thought: {step_thought.split(chr(10))[0]}')
        tool_name, result = react_step(step_thought, memory)
        if verbose and result:
            short = {k:(str(v)[:60]+'..' if len(str(v))>60 else v)
                     for k,v in result.items()} if isinstance(result,dict) else result
            print(f'Observation: {short}')
        observations[f'step_{step_i}'] = result

        # Carry latent & store facts
        if isinstance(result, dict):
            if 'latent' in result:
                prev_latent = result['latent']
                if 'chamfer_cd' in result:
                    memory.add_fact(f"Plant {result.get('plant_idx','?')} encoded: CD={result['chamfer_cd']:.4f}")
            elif 'yield_t_ha' in result:
                memory.add_fact(f"Yield: {result['yield_t_ha']} t/ha ({result['interpretation']}) +/-{result['uncertainty_std']}")
            elif 'trajectory' in result:
                r = result
                memory.add_fact(f"Interpolation {r['plant_a']}->{r['plant_b']}: yield range {r['yield_range']} t/ha")
            elif 'chamfer_distance' in result and 'yield_a_t_ha' in result:
                memory.add_fact(f"Compare {result['plant_a']} vs {result['plant_b']}: "
                                f"CD={result['chamfer_distance']:.4f}, "
                                f"yield_diff={result['yield_diff_t_ha']:+.3f} t/ha")

    # Synthesise final answer
    parts = [f"Analysis: '{query}'\n"]
    for _, result in observations.items():
        if not isinstance(result, dict) or 'error' in result: continue
        if 'yield_t_ha' in result:
            parts.append(f"Predicted yield: {result['yield_t_ha']} t/ha "
                         f"[95% CI: {result['confidence_interval']}] -- {result['interpretation']}")
        if 'trajectory' in result:
            parts.append(f"Interpolation {result['plant_a']} -> {result['plant_b']}:")
            for t in result['trajectory']:
                parts.append(f"  alpha={t['alpha']:.2f} -> {t['yield_t_ha']} t/ha (height={t['bbox_height']:.3f})")
        if 'chamfer_distance' in result and 'yield_a_t_ha' in result:
            parts.append(f"Plant {result['plant_a']}: {result['yield_a_t_ha']} t/ha | "
                         f"Plant {result['plant_b']}: {result['yield_b_t_ha']} t/ha | "
                         f"Cosine sim: {result['cosine_similarity']} | CD: {result['chamfer_distance']:.4f}")
        if 'latent_norm' in result:
            parts.append(f"Morphology: {result['interpretation']} "
                         f"(norm={result['latent_norm']}, pct={result['norm_percentile']}%)")
        if 'chamfer_cd' in result:
            parts.append(f"Plant {result['plant_idx']}: recon CD={result['chamfer_cd']:.4f}")

    final = '\n'.join(parts) if len(parts)>1 else 'No results.'
    memory.add_turn('assistant', final)
    if verbose:
        print('\n' + '='*60)
        print('FINAL ANSWER:')
        print(final)
        print('='*60)
        print(f'Memory: {memory}')
    return final

print('ReAct agent ready. Tools:', list(TOOLS.keys()))


ReAct agent ready. Tools: ['encode_plant', 'decode_latent', 'predict_yield', 'interpolate_latents', 'compare_plants', 'latent_statistics']


In [ ]:
# Run 4 demo queries that exercise all tools + memory

print('\n' + '#'*60)
print('DEMO 1: Yield prediction for plant 42')
print('#'*60)
answer1 = react_agent('What is the predicted yield for plant 42?', agent_memory)

print('\n' + '#'*60)
print('DEMO 2: Latent interpolation -- plant 10 to 99')
print('#'*60)
answer2 = react_agent(
    'Interpolate between plant 10 and plant 99 with 7 steps. '
    'Show how yield changes along the morphological trajectory.',
    agent_memory
)

print('\n' + '#'*60)
print('DEMO 3: Compare plants 7 and 200')
print('#'*60)
answer3 = react_agent(
    'Compare plant 7 versus plant 200 in terms of morphology and yield.',
    agent_memory
)

print('\n' + '#'*60)
print('DEMO 4: Full pipeline for plant 55 (memory-aware)')
print('#'*60)
answer4 = react_agent(
    'Analyse plant 55 completely: encode it, predict yield, '
    'and explain its morphological statistics.',
    agent_memory
)

print('\n' + '='*60)
print('MEMORY SUMMARY')
print('='*60)
print(f'Turns: {len(agent_memory.history)} | Facts: {len(agent_memory.facts)} | Tool calls: {len(agent_memory.tool_log)}')
print('\nAll stored facts:')
for f in agent_memory.facts:
    print(f'  [{f["time"]}] {f["fact"]}')



############################################################
DEMO 1: Yield prediction for plant 42
############################################################
QUERY: What is the predicted yield for plant 42?

--- Step 1 ---
Thought: Encode plant 42.
  >> Calling: encode_plant(['plant_idx'])
Observation: {'plant_idx': 42, 'latent': '[0.5398069024085999, 0.9815770387649536, 0.6175236105918884,..', 'chamfer_cd': 0.006864}

--- Step 2 ---
Thought: Predict yield.
  >> Calling: predict_yield(['latent'])
Observation: {'error': 'Expected more than 1 value per channel when training, got in..'}

--- Step 3 ---
Thought: Latent statistics.
  >> Calling: latent_statistics(['latent'])
Observation: {'latent_norm': 9.4314, 'norm_percentile': 24.7, 'mean_activation': 0.0074, 'std_activation': 0.5894, 'max_activation': 1.721, 'min_activation': -1.5698, 'cosine_to_mean_plant': 0.5847, 'interpretation': 'Typical plant morphology'}

FINAL ANSWER:
Analysis: 'What is the predicted yield for plant 42?'

Pla

In [ ]:
# ── Fixed tool_predict_yield ─────────────────────────────────────────
def tool_predict_yield(latent: list) -> dict:
    '''Predict maize yield (t/ha) + 95% CI via Monte Carlo dropout.'''
    z = torch.tensor(latent, dtype=torch.float32).unsqueeze(0).to(device)

    # MC-Dropout: keep BatchNorm in eval mode (needs >1 sample),
    # but enable Dropout layers for uncertainty sampling.
    yield_mlp.eval()
    for m in yield_mlp.modules():
        if isinstance(m, nn.Dropout):
            m.train()   # dropout ON  -> stochastic samples

    preds = []
    with torch.no_grad():
        for _ in range(20):
            preds.append(yield_mlp(z).item())

    yield_mlp.eval()   # restore fully to eval mode

    mu, std = float(np.mean(preds)), float(np.std(preds))
    return {
        'yield_t_ha':          round(mu, 3),
        'uncertainty_std':     round(std, 4),
        'confidence_interval': [round(mu - 2*std, 3), round(mu + 2*std, 3)],
        'interpretation':      ('High yield'   if mu > 3.5 else
                                'Medium yield' if mu > 2.5 else 'Low yield'),
    }


# ── Fixed interpolation cell ─────────────────────────────────────────
INTERP_A, INTERP_B, INTERP_STEPS = 10, 99, 5

z_a = torch.tensor(LATENT_INDEX[INTERP_A], dtype=torch.float32).to(device)
z_b = torch.tensor(LATENT_INDEX[INTERP_B], dtype=torch.float32).to(device)
alphas = np.linspace(0, 1, INTERP_STEPS)

fig_interp = make_subplots(
    rows=1, cols=INTERP_STEPS,
    specs=[[{'type': 'scatter3d'}] * INTERP_STEPS],
    subplot_titles=[f'alpha={a:.2f}' for a in alphas]
)

yield_vals = []
for col_i, alpha in enumerate(alphas, start=1):
    z_i = (1 - alpha) * z_a + alpha * z_b          # latent interpolation

    with torch.no_grad():                            # indented INSIDE the loop
        pts = ae_model.decode(z_i.unsqueeze(0))[0].cpu().numpy()

    y = tool_predict_yield(z_i.cpu().tolist())['yield_t_ha']  # uses fixed fn
    yield_vals.append(y)

    fig_interp.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers',
        marker=dict(size=1.5, color=pts[:, 2], colorscale='Plasma', showscale=False),
        showlegend=False
    ), row=1, col=col_i)

fig_interp.update_layout(
    title=f'Latent Interpolation: Plant {INTERP_A} -> Plant {INTERP_B}',
    template='plotly_dark', height=500, margin=dict(l=5, r=5, t=60, b=5)
)
fig_interp.show()

fig_yield_curve = go.Figure(go.Scatter(
    x=list(alphas), y=yield_vals, mode='lines+markers',
    line=dict(color='#00CC96', width=2), marker=dict(size=8)
))
fig_yield_curve.update_layout(
    title=f'Yield Trajectory: Plant {INTERP_A} -> Plant {INTERP_B}',
    xaxis_title='Interpolation alpha (0=plant A, 1=plant B)',
    yaxis_title='Predicted Yield (t/ha)', template='plotly_dark'
)
fig_yield_curve.show()
print('Yield along trajectory:', [f'{y:.3f}' for y in yield_vals])

Yield along trajectory: ['2.929', '3.363', '3.936', '3.826', '3.629']


In [ ]:
wandb.init(project='maize-dgcnn-agent', name='react-agent-results', resume='allow')

wandb.log({'interpolation_3d': wandb.Plotly(fig_interp)})
wandb.log({'yield_trajectory':  wandb.Plotly(fig_yield_curve)})

wandb.log({'tool_call_log': wandb.Table(
    columns=['time','tool','args','result_summary'],
    data=[[t['time'],t['tool'],str(t['args']),t['summary']] for t in agent_memory.tool_log]
)})

wandb.log({'agent_memory_facts': wandb.Table(
    columns=['time','fact'],
    data=[[f['time'],f['fact']] for f in agent_memory.facts]
)})

# Latent PCA coloured by yield
latent_2d = PCA(n_components=2).fit_transform(all_latents)
fig_pca = px.scatter(
    x=latent_2d[:,0], y=latent_2d[:,1], color=yield_labels,
    color_continuous_scale='Viridis',
    labels={'x':'PC1','y':'PC2','color':'Yield (t/ha)'},
    title='Latent Space PCA -- coloured by predicted yield',
    template='plotly_dark'
)
fig_pca.show()
wandb.log({'latent_pca_yield': wandb.Plotly(fig_pca)})

fig_yield_hist = px.histogram(
    x=yield_labels, nbins=40,
    title='Dataset Yield Distribution',
    labels={'x':'Yield (t/ha)','y':'Count'},
    template='plotly_dark', color_discrete_sequence=['#00CC96']
)
fig_yield_hist.show()
wandb.log({'yield_distribution': wandb.Plotly(fig_yield_hist)})

wandb.finish()
print('All results logged to W&B.')


All results logged to W&B.


In [ ]:
# Interactive REPL -- run this cell to chat with the agent
# Commands: 'quit', 'memory', 'clear'
print('ReAct Agent REPL')
print('Examples:')
print('  What is the yield for plant 5?')
print('  Interpolate between plant 3 and plant 150 with 6 steps')
print('  Compare plant 20 vs plant 80')
print('  Analyse plant 100 completely')
print()
while True:
    try:
        user_input = input('You: ').strip()
    except EOFError:
        break
    if not user_input: continue
    if user_input.lower() == 'quit':   print('Goodbye!'); break
    if user_input.lower() == 'memory':
        print(agent_memory.format_context())
        print(agent_memory)
        continue
    if user_input.lower() == 'clear':
        agent_memory.clear(); print('Memory cleared.'); continue
    react_agent(user_input, memory=agent_memory, verbose=True)


ReAct Agent REPL
Examples:
  What is the yield for plant 5?
  Interpolate between plant 3 and plant 150 with 6 steps
  Compare plant 20 vs plant 80
  Analyse plant 100 completely

You: What is the yield of plant 5?
QUERY: What is the yield of plant 5?

--- Step 1 ---
Thought: Encode plant 5.
  >> Calling: encode_plant(['plant_idx'])
Observation: {'plant_idx': 5, 'latent': '[-0.5844996571540833, -0.06123359501361847, 0.07297816872596..', 'chamfer_cd': 0.000856}

--- Step 2 ---
Thought: Predict yield.
  >> Calling: predict_yield(['latent'])
Observation: {'error': 'Expected more than 1 value per channel when training, got in..'}

--- Step 3 ---
Thought: Latent statistics.
  >> Calling: latent_statistics(['latent'])
Observation: {'latent_norm': 11.3497, 'norm_percentile': 82.2, 'mean_activation': -0.0457, 'std_activation': 0.7079, 'max_activation': 1.8021, 'min_activation': -1.9699, 'cosine_to_mean_plant': 0.5276, 'interpretation': 'Typical plant morphology'}

FINAL ANSWER:
Analysis: 'What